# Case 2 -- Transient, Single-Model, Freshwater Flow

There is recharge for first stress period, and then zero recharge for second stress period.

Two cases are presented here
1. One model layer
2. Three model layers


In [ ]:
from IPython.display import HTML
import pathlib as pl
import numpy as np
import matplotlib.pyplot as plt
import flopy
from swiutil import SwiAnimator

# path to mf6 executables with swi support: 
#   https://github.com/christianlangevin/modflow6-nightly-build/actions/workflows/nightly-build-swi.yml

# Put the name of the mf6 executable into mf6exe.txt,
# which is not under version control.
with open(pl.Path("./mf6exe.txt"), "r") as f:
    mf6exe = f.readline().strip()
print(f"using executable: {mf6exe}")

sim_ws = pl.Path("./temp/case2")

## Case 2a -- One Model Layer

In [ ]:
#create simple test model
Lx = 10000 # meters
delr, delc = 100., 1.
ncol = int(Lx / delr)
nlay = 1
nrow = 1
top = 50.
botm = -400.
recharge = {
    0: 0.001,
    1: 0.,
}
k = 10.
h0 = 0.
h1 = h0
icelltype = 1

name = 'mymodel'
sim = flopy.mf6.MFSimulation(
    sim_name=name, 
    sim_ws=sim_ws, 
    exe_name=mf6exe,
    memory_print_option="all"
)
nper = 2
nstp = 100
perlen = 200000.
perioddata = nper * [(perlen, nstp, 1.)]
tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)
ims = flopy.mf6.ModflowIms(
    sim, 
    print_option="summary",
    linear_acceleration="BICGSTAB",
)
gwf = flopy.mf6.ModflowGwf(
    sim, 
    modelname=name, 
    save_flows=True, 
    newtonoptions="newton",
)
dis = flopy.mf6.ModflowGwfdis(
    gwf, 
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
)
ic = flopy.mf6.ModflowGwfic(gwf, strt=0.)
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    alternative_cell_averaging=None,
    icelltype=icelltype,
    k=k,
)
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=1.e-5, 
    sy=0.2
)
zeta_file = name + '.zta'
swi = flopy.mf6.ModflowGwfswi(gwf, zeta_filerecord=zeta_file)
cghb = 1. * delr * delc / 10.
ghb = flopy.mf6.ModflowGwfghb(gwf, stress_period_data=[[0, 0, 0, h0, cghb],
                                                       [0, 0, ncol - 1, h1, cghb]])
rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)
budget_file = name + '.bud'
head_file = name + '.hds'
oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
        printrecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
)
sim.write_simulation()
sim.run_simulation()

In [ ]:
animator = SwiAnimator(sim=sim)
ani = animator.create()
HTML(ani.to_jshtml())

# Case 2b -- Three Model Layers

In [ ]:
#create simple test model
Lx = 10000 # meters
delr, delc = 100., 1.
ncol = int(Lx / delr)
nlay = 3
nrow = 1
top = 50.
botm = [-50, -200, -400.]
recharge = {
    0: 0.001,
    1: 0.,
}
k = [10., 10., 10.]
h0 = 0.
h1 = h0
icelltype = 1

name = 'mymodel'
sim = flopy.mf6.MFSimulation(
    sim_name=name, 
    sim_ws=sim_ws, 
    exe_name=mf6exe,
    memory_print_option="all",
    continue_=False,
)
nper = 2
nstp = 100
perlen = 200000.
perioddata = nper * [(perlen, nstp, 1.)]
tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)
ims = flopy.mf6.ModflowIms(
    sim, 
    print_option="summary",
    linear_acceleration="BICGSTAB",
)
gwf = flopy.mf6.ModflowGwf(
    sim, 
    modelname=name, 
    save_flows=True, 
    newtonoptions="newton",
)
dis = flopy.mf6.ModflowGwfdis(
    gwf, 
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
)
ic = flopy.mf6.ModflowGwfic(gwf, strt=0.)
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    alternative_cell_averaging=None,
    icelltype=icelltype,
    k=k,
)
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=0., 
    sy=0.2
)
zeta_file = name + '.zta'
swi = flopy.mf6.ModflowGwfswi(gwf, zeta_filerecord=zeta_file)
cghb = 1. * delr * delc / 10.
ghb = flopy.mf6.ModflowGwfghb(gwf, stress_period_data=[[0, 0, 0, h0, cghb],
                                                       [0, 0, ncol - 1, h1, cghb]])
rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)
budget_file = name + '.bud'
head_file = name + '.hds'
oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
        printrecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
)
sim.write_simulation()
sim.run_simulation()

In [ ]:
animator = SwiAnimator(sim=sim, edgecolor="none")
ani = animator.create()
HTML(ani.to_jshtml())

# Case 2c -- Three Model Layers with Well

In [ ]:
#create simple test model
Lx = 10000 # meters
delr, delc = 100., 1.
ncol = int(Lx / delr)
nlay = 3
nrow = 1
top = 50.
botm = [-50, -200, -400.]
recharge = {
    0: 0.001,
    # 1: 0.,
}
ghbspd = [[0, 0, 0, h0, cghb], [0, 0, ncol - 1, h1, cghb]]
welspd = [[0, 0, int(ncol/2), -10.]]
welspd = {0: None, 1: welspd}

k = [10., 10., 10.]
h0 = 0.
h1 = h0
icelltype = 1

name = 'mymodel'
sim = flopy.mf6.MFSimulation(
    sim_name=name, 
    sim_ws=sim_ws, 
    exe_name=mf6exe,
    memory_print_option="all",
    continue_=False,
)
nper = 2
nstp = 100
perlen = 200000.
perioddata = nper * [(perlen, nstp, 1.)]
tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)
ims = flopy.mf6.ModflowIms(
    sim, 
    print_option="summary",
    linear_acceleration="BICGSTAB",
)
gwf = flopy.mf6.ModflowGwf(
    sim, 
    modelname=name, 
    save_flows=True, 
    newtonoptions="newton",
)
dis = flopy.mf6.ModflowGwfdis(
    gwf, 
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
)
ic = flopy.mf6.ModflowGwfic(gwf, strt=0.)
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    alternative_cell_averaging=None,
    icelltype=icelltype,
    k=k,
)
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=0., 
    sy=0.2
)
zeta_file = name + '.zta'
swi = flopy.mf6.ModflowGwfswi(gwf, zeta_filerecord=zeta_file)
cghb = 1. * delr * delc / 10.
ghb = flopy.mf6.ModflowGwfghb(gwf, stress_period_data=ghbspd)
rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)
wel = flopy.mf6.ModflowGwfwel(gwf, stress_period_data=welspd)
budget_file = name + '.bud'
head_file = name + '.hds'
oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
        printrecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
)
sim.write_simulation()
sim.run_simulation()

In [ ]:
animator = SwiAnimator(sim=sim, edgecolor="none")
ani = animator.create()
HTML(ani.to_jshtml())